# fast_hrv Workspace

Bu notebook fast_hrv ekosistemini test etmek ve denemek için hazırlanmıştır.

**İçerik:**
1. ECG Sinyal Üretimi
2. fast_hrv (NumPy) ile İşleme
3. fast_hrv (Numba) ile İşleme
4. fast_hrv_rs (Rust Binding) ile İşleme
5. NeuroKit2 Karşılaştırma
6. Performans Benchmark
7. HRV Metrikleri Detaylı Analiz

## 1. Setup ve Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk
import time

# fast_hrv imports
from fast_hrv.ecg import process_ecg as process_ecg_numpy
from fast_hrv.peaks import ecg_peaks, find_peaks, fix_peaks
from fast_hrv.hrv import compute_hrv, compute_hrv_full
from fast_hrv import NUMBA_AVAILABLE

print(f"Numba available: {NUMBA_AVAILABLE}")

# Numba version (if available)
if NUMBA_AVAILABLE:
    from fast_hrv.ecg_numba import process_ecg as process_ecg_numba
    print("Numba version loaded")

# Rust binding (if available)
try:
    import fast_hrv_rs
    RUST_AVAILABLE = True
    print("Rust binding loaded")
except ImportError:
    RUST_AVAILABLE = False
    print("Rust binding not available. Install with: cd fast_hrv_rs && maturin develop --release")

## 2. ECG Sinyal Üretimi

In [ ]:
# ECG sinyal parametreleri
DURATION = 60  # saniye
SAMPLING_RATE = 512  # Hz
HEART_RATE = 70  # bpm

# Sentetik ECG üret
ecg_signal = nk.ecg_simulate(
    duration=DURATION,
    sampling_rate=SAMPLING_RATE,
    heart_rate=HEART_RATE,
    noise=0.05
)

print(f"ECG Signal Shape: {ecg_signal.shape}")
print(f"Duration: {DURATION}s, Sampling Rate: {SAMPLING_RATE}Hz")
print(f"Expected beats: ~{DURATION * HEART_RATE / 60:.0f}")

In [ ]:
# İlk 5 saniyeyi görselleştir
samples_5s = 5 * SAMPLING_RATE
time_axis = np.arange(samples_5s) / SAMPLING_RATE

plt.figure(figsize=(14, 4))
plt.plot(time_axis, ecg_signal[:samples_5s], 'b-', linewidth=0.8)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('ECG Signal (First 5 seconds)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. fast_hrv (NumPy) ile İşleme

In [ ]:
# NumPy versiyonu ile işle
result_numpy = process_ecg_numpy(ecg_signal, sampling_rate=SAMPLING_RATE, correct_artifacts=True)

print("=== fast_hrv (NumPy) Sonuçları ===")
print(f"R-peaks bulundu: {len(result_numpy['rpeaks'])}")
print(f"Mean HR: {result_numpy['stats']['mean_hr_bpm']:.1f} bpm")
print(f"SDNN: {result_numpy['stats']['sdnn_ms']:.2f} ms")
print(f"RMSSD: {result_numpy['stats']['rmssd_ms']:.2f} ms")
print(f"pNN50: {result_numpy['stats']['pnn50']:.2f}%")

In [ ]:
# R-peaks görselleştir
plt.figure(figsize=(14, 4))
plt.plot(time_axis, ecg_signal[:samples_5s], 'b-', linewidth=0.8, label='ECG')

# İlk 5 saniyedeki peaks
peaks_5s = result_numpy['rpeaks'][result_numpy['rpeaks'] < samples_5s]
plt.scatter(peaks_5s / SAMPLING_RATE, ecg_signal[peaks_5s], c='red', s=50, zorder=5, label='R-peaks')

plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('ECG with Detected R-peaks (NumPy)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. fast_hrv (Numba) ile İşleme

In [ ]:
if NUMBA_AVAILABLE:
    # İlk çağrı JIT compilation içerir (yavaş)
    print("JIT compilation (ilk çağrı)...")
    _ = process_ecg_numba(ecg_signal[:1000], sampling_rate=SAMPLING_RATE)
    
    # Gerçek işleme
    result_numba = process_ecg_numba(ecg_signal, sampling_rate=SAMPLING_RATE, correct_artifacts=True)
    
    print("\n=== fast_hrv (Numba) Sonuçları ===")
    print(f"R-peaks bulundu: {len(result_numba['rpeaks'])}")
    print(f"Mean HR: {result_numba['stats']['mean_hr_bpm']:.1f} bpm")
    print(f"SDNN: {result_numba['stats']['sdnn_ms']:.2f} ms")
    print(f"RMSSD: {result_numba['stats']['rmssd_ms']:.2f} ms")
    print(f"pNN50: {result_numba['stats']['pnn50']:.2f}%")
else:
    print("Numba not available. Install with: pip install numba")

## 5. fast_hrv_rs (Rust Binding) ile İşleme

In [ ]:
if RUST_AVAILABLE:
    result_rust = fast_hrv_rs.process_ecg(ecg_signal, SAMPLING_RATE, True)
    
    print("=== fast_hrv_rs (Rust) Sonuçları ===")
    print(f"R-peaks bulundu: {len(result_rust['rpeaks'])}")
    print(f"Mean HR: {result_rust['stats']['mean_hr_bpm']:.1f} bpm")
    print(f"SDNN: {result_rust['stats']['sdnn_ms']:.2f} ms")
    print(f"RMSSD: {result_rust['stats']['rmssd_ms']:.2f} ms")
    print(f"pNN50: {result_rust['stats']['pnn50']:.2f}%")
else:
    print("Rust binding not available.")
    print("Install with:")
    print("  cd fast_hrv_rs")
    print("  pip install maturin")
    print("  maturin develop --release")

## 6. NeuroKit2 Karşılaştırma

In [ ]:
# NeuroKit2 ile işle
nk_signals, nk_info = nk.ecg_process(ecg_signal, sampling_rate=SAMPLING_RATE)
nk_peaks = nk_info['ECG_R_Peaks']

# HRV hesapla
nk_hrv = nk.hrv_time(nk_peaks, sampling_rate=SAMPLING_RATE)

print("=== NeuroKit2 Sonuçları ===")
print(f"R-peaks bulundu: {len(nk_peaks)}")
print(f"Mean HR: {nk_hrv['HRV_MeanNN'].values[0] / 1000 * 60:.1f} bpm (from MeanNN)")
print(f"SDNN: {nk_hrv['HRV_SDNN'].values[0]:.2f} ms")
print(f"RMSSD: {nk_hrv['HRV_RMSSD'].values[0]:.2f} ms")
print(f"pNN50: {nk_hrv['HRV_pNN50'].values[0]:.2f}%")

In [ ]:
# Sonuçları karşılaştır
print("=" * 60)
print("SONUÇ KARŞILAŞTIRMASI")
print("=" * 60)
print(f"{'Metrik':<15} {'NeuroKit2':>12} {'NumPy':>12} {'Numba':>12} {'Rust':>12}")
print("-" * 60)

# N Peaks
numba_peaks = len(result_numba['rpeaks']) if NUMBA_AVAILABLE else '-'
rust_peaks = len(result_rust['rpeaks']) if RUST_AVAILABLE else '-'
print(f"{'N Peaks':<15} {len(nk_peaks):>12} {len(result_numpy['rpeaks']):>12} {numba_peaks:>12} {rust_peaks:>12}")

# SDNN
numba_sdnn = f"{result_numba['stats']['sdnn_ms']:.2f}" if NUMBA_AVAILABLE else '-'
rust_sdnn = f"{result_rust['stats']['sdnn_ms']:.2f}" if RUST_AVAILABLE else '-'
print(f"{'SDNN (ms)':<15} {nk_hrv['HRV_SDNN'].values[0]:>12.2f} {result_numpy['stats']['sdnn_ms']:>12.2f} {numba_sdnn:>12} {rust_sdnn:>12}")

# RMSSD
numba_rmssd = f"{result_numba['stats']['rmssd_ms']:.2f}" if NUMBA_AVAILABLE else '-'
rust_rmssd = f"{result_rust['stats']['rmssd_ms']:.2f}" if RUST_AVAILABLE else '-'
print(f"{'RMSSD (ms)':<15} {nk_hrv['HRV_RMSSD'].values[0]:>12.2f} {result_numpy['stats']['rmssd_ms']:>12.2f} {numba_rmssd:>12} {rust_rmssd:>12}")

# pNN50
numba_pnn50 = f"{result_numba['stats']['pnn50']:.2f}" if NUMBA_AVAILABLE else '-'
rust_pnn50 = f"{result_rust['stats']['pnn50']:.2f}" if RUST_AVAILABLE else '-'
print(f"{'pNN50 (%)':<15} {nk_hrv['HRV_pNN50'].values[0]:>12.2f} {result_numpy['stats']['pnn50']:>12.2f} {numba_pnn50:>12} {rust_pnn50:>12}")

## 7. Performans Benchmark

In [ ]:
# Uzun sinyal için benchmark
BENCH_DURATION = 180  # 3 dakika
ecg_long = nk.ecg_simulate(
    duration=BENCH_DURATION,
    sampling_rate=SAMPLING_RATE,
    heart_rate=HEART_RATE
)

N_RUNS = 5
results = {}

print(f"Benchmark: {BENCH_DURATION}s ECG, {N_RUNS} runs each")
print("=" * 50)

# NeuroKit2
times = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    signals, info = nk.ecg_process(ecg_long, sampling_rate=SAMPLING_RATE)
    nk.hrv_time(info['ECG_R_Peaks'], sampling_rate=SAMPLING_RATE)
    times.append((time.perf_counter() - start) * 1000)
results['NeuroKit2'] = np.mean(times)
print(f"NeuroKit2:  {results['NeuroKit2']:.1f} ms")

# NumPy
times = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    process_ecg_numpy(ecg_long, sampling_rate=SAMPLING_RATE, correct_artifacts=True)
    times.append((time.perf_counter() - start) * 1000)
results['NumPy'] = np.mean(times)
print(f"NumPy:      {results['NumPy']:.1f} ms ({results['NeuroKit2']/results['NumPy']:.0f}x faster)")

# Numba
if NUMBA_AVAILABLE:
    # Warmup
    process_ecg_numba(ecg_long[:1000], sampling_rate=SAMPLING_RATE)
    
    times = []
    for _ in range(N_RUNS):
        start = time.perf_counter()
        process_ecg_numba(ecg_long, sampling_rate=SAMPLING_RATE, correct_artifacts=True)
        times.append((time.perf_counter() - start) * 1000)
    results['Numba'] = np.mean(times)
    print(f"Numba:      {results['Numba']:.1f} ms ({results['NeuroKit2']/results['Numba']:.0f}x faster)")

# Rust
if RUST_AVAILABLE:
    times = []
    for _ in range(N_RUNS):
        start = time.perf_counter()
        fast_hrv_rs.process_ecg(ecg_long, SAMPLING_RATE, True)
        times.append((time.perf_counter() - start) * 1000)
    results['Rust'] = np.mean(times)
    print(f"Rust:       {results['Rust']:.1f} ms ({results['NeuroKit2']/results['Rust']:.0f}x faster)")

In [ ]:
# Benchmark sonuçlarını görselleştir
labels = list(results.keys())
values = list(results.values())
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
bars1 = ax1.bar(labels, values, color=colors[:len(labels)])
ax1.set_ylabel('Time (ms)')
ax1.set_title('Processing Time (Linear Scale)')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:.1f} ms', ha='center', va='bottom', fontsize=10)

# Log scale
bars2 = ax2.bar(labels, values, color=colors[:len(labels)])
ax2.set_ylabel('Time (ms) - Log Scale')
ax2.set_title('Processing Time (Log Scale)')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. HRV Metrikleri Detaylı Analiz

In [ ]:
# RR intervals hesapla
rr_intervals = np.diff(result_numpy['rpeaks']) / SAMPLING_RATE * 1000  # ms cinsinden

print(f"RR Intervals: {len(rr_intervals)} adet")
print(f"Mean RR: {np.mean(rr_intervals):.1f} ms")
print(f"Std RR: {np.std(rr_intervals, ddof=1):.1f} ms")
print(f"Min RR: {np.min(rr_intervals):.1f} ms")
print(f"Max RR: {np.max(rr_intervals):.1f} ms")

In [ ]:
# Tüm HRV metrikleri
hrv_full = compute_hrv_full(rr_intervals)

print("=== Time Domain Metrics ===")
print(f"SDNN:  {hrv_full['sdnn']:.2f} ms")
print(f"RMSSD: {hrv_full['rmssd']:.2f} ms")
print(f"pNN50: {hrv_full['pnn50']:.2f}%")
print(f"pNN20: {hrv_full['pnn20']:.2f}%")
print(f"SDSD:  {hrv_full['sdsd']:.2f} ms")

print("\n=== Nonlinear Metrics ===")
print(f"ApEn:   {hrv_full['apen']:.4f}")
print(f"SampEn: {hrv_full['sampen']:.4f}")
print(f"DFA α1: {hrv_full['dfa_alpha1']:.4f}")
print(f"DFA α2: {hrv_full['dfa_alpha2']:.4f}")

print("\n=== Poincare Metrics ===")
print(f"SD1: {hrv_full['sd1']:.2f} ms")
print(f"SD2: {hrv_full['sd2']:.2f} ms")
print(f"SD1/SD2: {hrv_full['sd1_sd2_ratio']:.4f}")
print(f"CSI: {hrv_full['csi']:.4f}")
print(f"CVI: {hrv_full['cvi']:.4f}")

In [ ]:
# Poincare Plot
rr_n = rr_intervals[:-1]
rr_n1 = rr_intervals[1:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Poincare scatter
ax1.scatter(rr_n, rr_n1, alpha=0.5, s=20)
ax1.plot([min(rr_n), max(rr_n)], [min(rr_n), max(rr_n)], 'r--', alpha=0.5)
ax1.set_xlabel('RR(n) [ms]')
ax1.set_ylabel('RR(n+1) [ms]')
ax1.set_title(f"Poincare Plot (SD1={hrv_full['sd1']:.1f}, SD2={hrv_full['sd2']:.1f})")
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# RR interval distribution
ax2.hist(rr_intervals, bins=30, edgecolor='black', alpha=0.7)
ax2.axvline(np.mean(rr_intervals), color='r', linestyle='--', label=f'Mean: {np.mean(rr_intervals):.1f} ms')
ax2.set_xlabel('RR Interval [ms]')
ax2.set_ylabel('Count')
ax2.set_title('RR Interval Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# RR Tachogram
beat_times = result_numpy['rpeaks'][1:] / SAMPLING_RATE  # saniye

plt.figure(figsize=(14, 4))
plt.plot(beat_times, rr_intervals, 'b-', linewidth=0.8)
plt.fill_between(beat_times, rr_intervals, alpha=0.3)
plt.axhline(np.mean(rr_intervals), color='r', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(rr_intervals):.1f} ms')
plt.xlabel('Time (s)')
plt.ylabel('RR Interval (ms)')
plt.title('RR Tachogram')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Rust Binding Detaylı Kullanım

In [ ]:
if RUST_AVAILABLE:
    print("=== Rust Binding Fonksiyonları ===")
    
    # ECG clean
    cleaned = fast_hrv_rs.ecg_clean(ecg_signal, SAMPLING_RATE)
    print(f"ecg_clean: {cleaned.shape}")
    
    # Find peaks
    peaks = fast_hrv_rs.find_peaks(cleaned, SAMPLING_RATE)
    print(f"find_peaks: {len(peaks)} peaks")
    
    # Fix peaks
    fixed_peaks, artifacts = fast_hrv_rs.fix_peaks(peaks, SAMPLING_RATE)
    print(f"fix_peaks: {len(fixed_peaks)} peaks")
    print(f"  - Ectopic: {artifacts['ectopic']}")
    print(f"  - Missed: {artifacts['missed']}")
    print(f"  - Extra: {artifacts['extra']}")
    
    # HRV metrics
    rr = np.diff(fixed_peaks) / SAMPLING_RATE * 1000
    
    print(f"\n=== HRV Metrics (Rust) ===")
    print(f"RMSSD: {fast_hrv_rs.rmssd(rr):.2f} ms")
    print(f"SDNN:  {fast_hrv_rs.sdnn(rr):.2f} ms")
    print(f"pNN50: {fast_hrv_rs.pnn50(rr):.2f}%")
    print(f"pNN20: {fast_hrv_rs.pnn20(rr):.2f}%")
    
    # All metrics at once
    all_metrics = fast_hrv_rs.hrv_metrics(rr)
    print(f"\n=== All HRV Metrics ===")
    for key, value in all_metrics.items():
        print(f"{key}: {value:.4f}")
else:
    print("Rust binding not available")

## Sonuç

Bu notebook'ta fast_hrv ekosisteminin tüm versiyonlarını test ettik:

1. **NumPy**: Pure Python, bağımlılık az, debug kolay
2. **Numba**: JIT compiled, ~100x hızlı
3. **Rust Binding**: En hızlı (~500x), Python API

Tüm versiyonlar aynı sonuçları üretiyor ve NeuroKit2 ile uyumlu.